# FunSearch + RAG for the Capacitated Vehicle Routing Problem (CVRP)

## Overview

This notebook reproduces the **FunSearch + RAG** experiment pipeline for solving the **Capacitated Vehicle Routing Problem (CVRP)**.

### Task

The **CVRP** is a classic combinatorial optimisation problem:  
given a depot and a set of customers with known demands, find the minimum-cost set of vehicle routes such that every customer is visited exactly once and no route exceeds the vehicle's capacity.

### Method

We augment **FunSearch** — an evolutionary large-language-model (LLM) program search framework originally proposed by DeepMind — with a **Retrieval-Augmented Generation (RAG)** module:

| Component | Role |
|-----------|------|
| **FunSearch core** | Iteratively prompts an LLM to evolve a Python heuristic for CVRP, keeping the best programs in an island-based population |
| **RAG module** | Before each LLM call, retrieves the most relevant chunks from a curated knowledge corpus and injects them into the prompt, giving the model algorithmic context it would not otherwise have |
| **Evaluator** | Executes the generated heuristic on held-out CVRP benchmark instances (CVRPLIB Set-B) and returns a scalar score (negative total route distance) |

### Evaluation

Results are assessed on **CVRPLIB Set-B** benchmark instances. Key metrics reported:

- `best_score` — best negative total distance achieved across all islands
- `valid_eval_ratio` — fraction of LLM-generated programs that compiled and ran successfully
- `retrieval_events` — number of RAG retrievals performed
- `retrieval_mean_confidence` — mean semantic-similarity confidence of retrieved chunks

### Reproducibility

Each run clones a fixed GitHub branch, pins dependency versions, and logs all outputs to a timestamped directory under `results/experiments/`.

---
> **Budget control:** The default `VERIFICATION_BUDGET = 2` keeps cost low for smoke-testing.  
> Increase it (e.g. to `100`) for a full experiment run.

In [ ]:
"""
Cell 1 – Repository Setup
--------------------------
Clones the project from GitHub on a fresh Colab runtime, or pulls the latest
changes if the runtime is being reused. This ensures the notebook always runs
against the pinned RAG branch, making results fully reproducible.
"""
import subprocess
import os

# ── Source of truth: repository URL and branch ────────────────────────────────
REPO_URL = "https://github.com/Zz1jd/CSProjectAI.git"
BRANCH = "RAG"
REPO_DIR = "/content/CSProjectAI"

# Clone on first run; pull on subsequent runs within the same Colab session.
if not os.path.exists(REPO_DIR):
    print("Cloning repository (first run)...")
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO_DIR],
        check=True,
    )
    print("Repository cloned successfully.")
else:
    print("Repository already exists – pulling latest changes...")
    os.chdir(REPO_DIR)
    subprocess.run(["git", "pull", "origin", BRANCH], check=True)
    print("Repository updated successfully.")

# Make the project root the working directory so all relative imports resolve.
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# Log the exact commit hash for reproducibility tracing.
git_head = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
).strip()
print(f"Branch : {BRANCH}")
print(f"Commit : {git_head}")

In [ ]:
# Cell 2 – Install Dependencies
# ─────────────────────────────
# vrplib  : reads CVRPLIB benchmark files (.vrp / .sol format)
# numpy   : numerical operations inside the generated heuristics
# openai  : Python client used to call the LLM API (OpenAI-compatible endpoint)
#
# The -q flag suppresses verbose pip output to keep the Colab log clean.
!pip install -q vrplib numpy openai

## API Credentials

The pipeline calls two external APIs:

| Variable | Purpose |
|----------|---------|
| `FUNSEARCH_API_KEY` | LLM chat-completion API (OpenAI-compatible) |
| `FUNSEARCH_EMBEDDING_API_KEY` | Embedding API used by the RAG retriever |

**Option A – Set directly in the cell below** (easiest for Colab; keep the notebook private).  
**Option B – Mount Google Drive and load a `.env` file:**
```python
from google.colab import drive
drive.mount('/content/drive')
# Then place credentials in /content/drive/MyDrive/funsearch.env
```
**Option C – Use Colab Secrets** (Extras → Secrets) and read with `google.colab.userdata`.

> ⚠️ Never commit real API keys to a public repository.

In [ ]:
# Cell 3 – API Credentials
# ─────────────────────────
# Set credentials as environment variables so the rest of the pipeline can read
# them via os.environ without any hard-coded secrets in the source modules.
#
# Replace the placeholder strings with your actual keys before running.
import os

os.environ["FUNSEARCH_API_KEY"] = "sk-vWpzPgcJaoamJOr998VvL5H4Z2uTt6jNmPk0SftpmCQJYZ5C"
os.environ["FUNSEARCH_EMBEDDING_API_KEY"] = "sk-jwbxcbszdqdinhqofxikohzyjisdvwnkljbrzkfqufuxcbyy"

# Confirm variables are registered (prints True/False, not the key values).
print("FUNSEARCH_API_KEY set         :", bool(os.environ.get("FUNSEARCH_API_KEY")))
print("FUNSEARCH_EMBEDDING_API_KEY set:", bool(os.environ.get("FUNSEARCH_EMBEDDING_API_KEY")))

## Dataset & Problem Description

### Capacitated Vehicle Routing Problem (CVRP)

The **CVRP** asks: given a depot node `0` and `n` customer nodes, each with a demand $d_i$, find a minimum-cost set of routes for a fleet of vehicles with uniform capacity $Q$ such that:
- Every customer is visited exactly once.
- The total demand on each route does not exceed $Q$.
- Every route starts and ends at the depot.

Formally, the objective is:

$$\min \sum_{k} \sum_{(i,j) \in \text{route}_k} c_{ij}$$

where $c_{ij}$ is the Euclidean distance between nodes $i$ and $j$.

### Benchmark: CVRPLIB Set-B

We evaluate on **CVRPLIB Set-B** (Christofides & Eilon, 1969) — 23 standardised instances ranging from 31 to 78 customers. The dataset is stored in the repository under `cvrplib/setB/` in the standard `.vrp` / `.sol` format. It is loaded by `dataset.py → load_cvrp_dataset()`.

The evaluator scores a candidate heuristic by executing it on these instances and returning the **negative total distance** (higher is better, so FunSearch maximises this).

### Knowledge Corpus for RAG

The RAG module draws from a curated corpus (`corpus/`) of chunked algorithmic knowledge — VRP heuristics, savings algorithms, and neighbourhood-search descriptions — stored as text chunks. At each sampling step the retriever finds the most semantically similar chunks to the current prompt context and injects them as additional context for the LLM.

In [ ]:
# Cell 4 – Import Experiment Helpers
# ────────────────────────────────────
# All project modules live under the cloned repository root (REPO_DIR).
# The run_rag_eval script exposes three public symbols:
#
#   ExperimentRunConfig  – immutable dataclass controlling runtime knobs
#                          (seed, run_mode, budget, output directories)
#   ModelSpec            – names the LLM model under test and the output label
#                          used to name the results directory
#   run_rag_eval()       – orchestrates a single RAG experiment end-to-end:
#                          builds runtime config → runs FunSearch → parses logs
#                          → writes a JSON summary → returns a summary dict
from pathlib import Path

from scripts.run_rag_eval import ExperimentRunConfig
from scripts.run_rag_eval import ModelSpec
from scripts.run_rag_eval import run_rag_eval

print("Imports OK – experiment helpers loaded.")

In [ ]:
# Cell 5 – Experiment Configuration
# ───────────────────────────────────
# VERIFICATION_BUDGET controls how many LLM sampling steps FunSearch performs.
#   • 2   → cheap smoke-test (confirms the pipeline runs end-to-end)
#   • 100 → full experiment (recommended for final results)
#
# ExperimentRunConfig knobs:
#   budget     – maximum number of LLM samples (≈ API calls)
#   run_mode   – "stage_eval": evaluate each program on all dataset instances
#   seed       – global random seed for reproducibility
#
# ModelSpec knobs:
#   model_name   – OpenAI-compatible model identifier sent to the API
#   result_label – used as part of the timestamped output directory name

VERIFICATION_BUDGET = 2  # ← increase to 100 for a full run

experiment_config = ExperimentRunConfig(budget=VERIFICATION_BUDGET)

llm_model = ModelSpec(
    model_name="gpt-3.5-turbo",
    result_label="test_gpt_3_5_turbo",
)

print(f"Budget   : {experiment_config.budget} samples")
print(f"Run mode : {experiment_config.run_mode}")
print(f"Seed     : {experiment_config.seed}")
print(f"Model    : {llm_model.model_name}")
print(f"Label    : {llm_model.result_label}")

In [ ]:
# Cell 6 – Run Experiment
# ────────────────────────
# run_rag_eval() executes the full FunSearch + RAG pipeline:
#
#   1. Loads CVRPLIB Set-B instances from cvrplib/setB/ via dataset.py
#   2. Builds the runtime config with RAG enabled (hybrid retrieval, top-k=2)
#   3. Runs FunSearch for `budget` sampling steps:
#        a. Selects parent programs from the island population
#        b. Retrieves relevant corpus chunks (RAG)
#        c. Constructs a Chain-of-Thought prompt and calls the LLM
#        d. Parses and sandboxes the returned Python function
#        e. Evaluates on CVRP instances; updates island scores
#   4. Writes a log file (rag.txt) and a JSON summary (rag_summary.json)
#      under results/experiments/<timestamp>_<label>_rag/
#   5. Returns the summary dict for immediate inspection below

print("Starting experiment... (this may take a few minutes for budget > 2)")
experiment_summary = run_rag_eval(
    experiment_config=experiment_config,
    model_spec=llm_model,
)
print("Experiment complete.")
experiment_summary

In [ ]:
# Cell 7 – Results
# ─────────────────
# Parse and display the key metrics from the completed experiment.
#
# Metric glossary:
#   summary_path            – path to the JSON summary file on disk
#   log_path                – path to the raw run log (rag.txt)
#   best_score              – highest scoring program found (negative total
#                             distance; less negative = better routing)
#   valid_eval_ratio        – fraction of LLM outputs that were syntactically
#                             valid Python AND produced a finite route score
#                             (higher → LLM generates more usable code)
#   retrieval_events        – total number of RAG retrievals performed
#   retrieval_mean_confidence – mean cosine-similarity score of retrieved
#                              chunks (higher → more on-topic context injected)

summary_path = Path(experiment_summary["summary_path"])
run_result = experiment_summary["run"]

print("─" * 50)
print("Experiment Results")
print("─" * 50)
print(f"  Output directory  : {experiment_summary['experiment_dir']}")
print(f"  Summary JSON      : {summary_path}")
print(f"  Raw log           : {run_result['log_path']}")
print("─" * 50)
print(f"  best_score              : {run_result['best']}")
print(f"  valid_eval_ratio        : {run_result['valid_eval_ratio']}")
print(f"  retrieval_events        : {run_result['retrieval_events']}")
print(f"  retrieval_mean_confidence: {run_result['retrieval_mean_confidence']}")
print("─" * 50)

# Show full summary dict for complete traceability.
import json
print("\nFull summary:")
print(json.dumps(experiment_summary, indent=2))